# Retail Sales Analysis

Sales EDA using Spark SQL and PySpark in Databricks. Based on the Kaggle *Store Sales – Time Series Forecasting* dataset and the cleaned Delta tables from Notebook 02.

**Note:** `sales` is quantity, not revenue; quantities from different product families are not always directly comparable.

## 1. Stores and cities

Compare city totals and average daily sales per store.

In [0]:

%sql

SELECT
    s.date,
    s.store_nbr,
    s.family,
    s.sales,
    st.city,
    st.state,
    st.type
FROM workspace.default.sales_cleaned AS s
INNER JOIN workspace.default.stores_cleaned AS st
    ON s.store_nbr = st.store_nbr
WHERE s.date = '2017-08-15'
    AND s.family = 'BEVERAGES'
ORDER BY s.store_nbr
LIMIT 10;

date,store_nbr,family,sales,city,state,type
2017-08-15,1,BEVERAGES,1942.0,Quito,Pichincha,D
2017-08-15,2,BEVERAGES,2093.0,Quito,Pichincha,D
2017-08-15,3,BEVERAGES,6915.0,Quito,Pichincha,D
2017-08-15,4,BEVERAGES,1739.0,Quito,Pichincha,D
2017-08-15,5,BEVERAGES,1654.0,Santo Domingo,Santo Domingo de los Tsachilas,D
2017-08-15,6,BEVERAGES,2678.0,Quito,Pichincha,D
2017-08-15,7,BEVERAGES,3999.0,Quito,Pichincha,D
2017-08-15,8,BEVERAGES,3626.0,Quito,Pichincha,D
2017-08-15,9,BEVERAGES,3530.0,Quito,Pichincha,B
2017-08-15,10,BEVERAGES,1608.0,Quito,Pichincha,C


In [0]:
%sql

SELECT
    st.city,
    COUNT(DISTINCT s.store_nbr) AS store_count,
    ROUND(SUM(s.sales), 2) AS total_sales
FROM workspace.default.sales_cleaned AS s
INNER JOIN workspace.default.stores_cleaned AS st
    ON s.store_nbr = st.store_nbr
GROUP BY st.city
ORDER BY total_sales DESC
LIMIT 10;


city,store_count,total_sales
Quito,18,5.5674183683E8
Guayaquil,8,1.2296726048E8
Cuenca,3,4.916859603E7
Ambato,2,4.03044013E7
Santo Domingo,3,3.583432304E7
Machala,2,3.338679654E7
Cayambe,1,2.832555848E7
Latacunga,2,2.118226378E7
Daule,1,1.918802138E7
Loja,1,1.885999926E7


In [0]:
%sql

SELECT
    st.city,
    COUNT(DISTINCT s.store_nbr) AS store_count,
    ROUND(SUM(s.sales), 2) AS total_sales,
    ROUND(
        SUM(s.sales) / COUNT(DISTINCT s.store_nbr), 2
    ) AS avg_sales_per_store
FROM workspace.default.sales_cleaned AS s
INNER JOIN workspace.default.stores_cleaned AS st
    ON s.store_nbr = st.store_nbr
GROUP BY st.city
ORDER BY avg_sales_per_store DESC
LIMIT 10;


city,store_count,total_sales,avg_sales_per_store
Quito,18,5.5674183683E8,3.093010205E7
Cayambe,1,2.832555848E7,2.832555848E7
Ambato,2,4.03044013E7,2.015220065E7
Daule,1,1.918802138E7,1.918802138E7
Loja,1,1.885999926E7,1.885999926E7
Babahoyo,1,1.77468727E7,1.77468727E7
Machala,2,3.338679654E7,1.669339827E7
Esmeraldas,1,1.639169221E7,1.639169221E7
Cuenca,3,4.916859603E7,1.638953201E7
Guayaquil,8,1.2296726048E8,1.537090756E7


In [0]:
%sql

WITH store_sales AS (
    SELECT
        s.store_nbr,
        st.city,
        SUM(s.sales) AS total_sales,
        COUNT(DISTINCT s.date) AS days_with_data
    FROM workspace.default.sales_cleaned AS s
    INNER JOIN workspace.default.stores_cleaned AS st
        ON s.store_nbr = st.store_nbr
    GROUP BY s.store_nbr, st.city
)

SELECT
    city,
    COUNT(*) AS store_count,
    ROUND(
        AVG(total_sales / days_with_data), 2
    ) AS avg_daily_sales_per_store
FROM store_sales
GROUP BY city
ORDER BY avg_daily_sales_per_store DESC
LIMIT 10;


city,store_count,avg_daily_sales_per_store
Quito,18,18367.04
Cayambe,1,16820.4
Ambato,2,11966.86
Daule,1,11394.31
Loja,1,11199.52
Babahoyo,1,10538.52
Machala,2,9912.94
Esmeraldas,1,9733.78
Cuenca,3,9732.5
Guayaquil,8,9127.62


## 2. Time trends and seasonality

Explore monthly trends, then normalize each year's monthly averages to compare seasonality.

In [0]:
%sql

SELECT
    DATE_TRUNC('MONTH', date) AS sales_month,
    ROUND(SUM(sales), 2) AS total_sales,
    COUNT(DISTINCT date) AS days_with_data,
    ROUND(
        SUM(sales) / COUNT(DISTINCT date), 2
    ) AS avg_daily_sales
FROM workspace.default.sales_cleaned
GROUP BY DATE_TRUNC('MONTH', date)
ORDER BY sales_month;


sales_month,total_sales,days_with_data,avg_daily_sales
2013-01-01T00:00:00.000Z,1.032762474E7,31,333149.19
2013-02-01T00:00:00.000Z,9658959.78,28,344962.85
2013-03-01T00:00:00.000Z,1.142849704E7,31,368661.19
2013-04-01T00:00:00.000Z,1.099346474E7,30,366448.82
2013-05-01T00:00:00.000Z,1.159770401E7,31,374119.48
2013-06-01T00:00:00.000Z,1.168934406E7,30,389644.8
2013-07-01T00:00:00.000Z,1.125740061E7,31,363141.96
2013-08-01T00:00:00.000Z,1.173778892E7,31,378638.35
2013-09-01T00:00:00.000Z,1.179293323E7,30,393097.77
2013-10-01T00:00:00.000Z,1.177562036E7,31,379858.72


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    year,
    month,
    ROUND(
        SUM(sales) / COUNT(DISTINCT date), 2
    ) AS avg_daily_sales
FROM workspace.default.sales_cleaned
GROUP BY year, month
ORDER BY year, month;


year,month,avg_daily_sales
2013,1,333149.19
2013,2,344962.85
2013,3,368661.19
2013,4,366448.82
2013,5,374119.48
2013,6,389644.8
2013,7,363141.96
2013,8,378638.35
2013,9,393097.77
2013,10,379858.72


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

WITH monthly_sales AS (
    SELECT
        year,
        month,
        SUM(sales) / COUNT(DISTINCT date) AS avg_daily_sales
    FROM workspace.default.sales_cleaned
    WHERE year BETWEEN 2013 AND 2016
    GROUP BY year, month
)

SELECT
    year,
    month,
    ROUND(
        avg_daily_sales /
        AVG(avg_daily_sales) OVER (PARTITION BY year) * 100,
        2
    ) AS sales_index
FROM monthly_sales
ORDER BY year, month;


year,month,sales_index
2013,1,86.34
2013,2,89.4
2013,3,95.54
2013,4,94.97
2013,5,96.96
2013,6,100.98
2013,7,94.11
2013,8,98.13
2013,9,101.87
2013,10,98.44


Databricks visualization. Run in Databricks to view.

## 3. Product categories

Identify high-volume categories and compare beverage sales across cities.

In [0]:
%sql

SELECT
    family,
    ROUND(SUM(sales), 2) AS total_sales,
    COUNT(DISTINCT date) AS days_with_data,
    ROUND(
        SUM(sales) / COUNT(DISTINCT date), 2
    ) AS avg_daily_sales
FROM workspace.default.sales_cleaned
GROUP BY family
ORDER BY total_sales DESC
LIMIT 10;


family,total_sales,days_with_data,avg_daily_sales
GROCERY I,3.4346273489E8,1684,203956.49
BEVERAGES,2.16954486E8,1684,128832.83
PRODUCE,1.2270468468E8,1684,72865.01
CLEANING,9.7521289E7,1684,57910.5
DAIRY,6.4487709E7,1684,38294.36
BREAD/BAKERY,4.213394558E7,1684,25020.16
POULTRY,3.187600447E7,1684,18928.74
MEATS,3.10864684E7,1684,18459.9
PERSONAL CARE,2.4592051E7,1684,14603.36
DELI,2.411032247E7,1684,14317.29


In [0]:
%sql

WITH store_beverage_sales AS (
    SELECT
        s.store_nbr,
        st.city,
        SUM(s.sales) AS total_sales,
        COUNT(DISTINCT s.date) AS days_with_data
    FROM workspace.default.sales_cleaned AS s
    INNER JOIN workspace.default.stores_cleaned AS st
        ON s.store_nbr = st.store_nbr
    WHERE s.family = 'BEVERAGES'
    GROUP BY s.store_nbr, st.city
)

SELECT
    city,
    COUNT(*) AS store_count,
    ROUND(
        AVG(total_sales / days_with_data), 2
    ) AS avg_daily_beverage_sales
FROM store_beverage_sales
GROUP BY city
ORDER BY avg_daily_beverage_sales DESC
LIMIT 10;


city,store_count,avg_daily_beverage_sales
Quito,18,3745.22
Cayambe,1,3645.67
Esmeraldas,1,2972.41
Daule,1,2404.89
Ambato,2,2197.95
Libertad,1,1962.1
Guaranda,1,1957.76
Machala,2,1912.55
Guayaquil,8,1877.35
Cuenca,3,1823.82


## 4. Promotions

Compare beverage sales by promotion status and promotion count, then check differences within years and stores.

In [0]:
%sql

SELECT
    CASE
        WHEN onpromotion > 0 THEN 'Promotion'
        ELSE 'No Promotion'
    END AS promotion_status,
    COUNT(*) AS record_count,
    ROUND(AVG(sales), 2) AS avg_sales,
    ROUND(AVG(onpromotion), 2) AS avg_promoted_items
FROM workspace.default.sales_cleaned
WHERE family = 'BEVERAGES'
GROUP BY
    CASE
        WHEN onpromotion > 0 THEN 'Promotion'
        ELSE 'No Promotion'
    END
ORDER BY promotion_status;

promotion_status,record_count,avg_sales,avg_promoted_items
No Promotion,39231,1292.27,0.0
Promotion,51705,3215.5,17.54


In [0]:
%sql

WITH promotion_groups AS (
    SELECT
        sales,
        onpromotion,
        CASE
            WHEN onpromotion = 0 THEN 'No Promotion'
            WHEN onpromotion <= 5 THEN 'Low'
            WHEN onpromotion <= 20 THEN 'Medium'
            ELSE 'High'
        END AS promotion_level
    FROM workspace.default.sales_cleaned
    WHERE family = 'BEVERAGES'
)

SELECT
    promotion_level,
    COUNT(*) AS record_count,
    ROUND(AVG(onpromotion), 2) AS avg_promoted_items,
    ROUND(AVG(sales), 2) AS avg_sales
FROM promotion_groups
GROUP BY promotion_level
ORDER BY MIN(onpromotion);


promotion_level,record_count,avg_promoted_items,avg_sales
No Promotion,39231,0.0,1292.27
Low,17707,2.36,2635.9
Medium,16388,12.31,3059.77
High,17610,37.67,3943.21


In [0]:
%sql

WITH promotion_groups AS (
    SELECT
        year,
        sales,
        onpromotion,
        CASE
            WHEN onpromotion = 0 THEN 'No Promotion'
            WHEN onpromotion <= 5 THEN 'Low'
            WHEN onpromotion <= 20 THEN 'Medium'
            ELSE 'High'
        END AS promotion_level
    FROM workspace.default.sales_cleaned
    WHERE family = 'BEVERAGES'
)

SELECT
    year,
    promotion_level,
    COUNT(*) AS record_count,
    ROUND(AVG(sales), 2) AS avg_sales
FROM promotion_groups
GROUP BY year, promotion_level
ORDER BY year, MIN(onpromotion);


year,promotion_level,record_count,avg_sales
2013,No Promotion,19656,1107.95
2014,No Promotion,12460,1632.7
2014,Low,6321,2793.21
2014,Medium,871,3486.87
2014,High,4,3515.25
2015,No Promotion,5444,1163.12
2015,Low,9149,2544.68
2015,Medium,3982,3369.59
2015,High,1081,4990.68
2016,No Promotion,1488,1450.7


In [0]:
%sql

WITH store_sales AS (
    SELECT
        sales,
        CASE
            WHEN onpromotion > 0 THEN 'Promotion'
            ELSE 'No Promotion'
        END AS promotion_status
    FROM workspace.default.sales_cleaned
    WHERE family = 'BEVERAGES'
        AND year = 2015
        AND store_nbr = 1
)

SELECT
    promotion_status,
    COUNT(*) AS days_with_data,
    ROUND(AVG(sales), 2) AS avg_sales
FROM store_sales
GROUP BY promotion_status
ORDER BY promotion_status;


promotion_status,days_with_data,avg_sales
No Promotion,67,1068.96
Promotion,297,1704.58


In [0]:
%sql

SELECT
    store_nbr,
    COUNT(CASE WHEN onpromotion = 0 THEN 1 END) AS no_promo_days,
    COUNT(CASE WHEN onpromotion > 0 THEN 1 END) AS promo_days,
    ROUND(
        AVG(CASE WHEN onpromotion = 0 THEN sales END), 2
    ) AS no_promo_avg,
    ROUND(
        AVG(CASE WHEN onpromotion > 0 THEN sales END), 2
    ) AS promo_avg
FROM workspace.default.sales_cleaned
WHERE family = 'BEVERAGES'
    AND year = 2015
GROUP BY store_nbr
HAVING
    COUNT(CASE WHEN onpromotion = 0 THEN 1 END) >= 20
    AND COUNT(CASE WHEN onpromotion > 0 THEN 1 END) >= 20
ORDER BY store_nbr
LIMIT 10;


store_nbr,no_promo_days,promo_days,no_promo_avg,promo_avg
1,67,297,1068.96,1704.58
2,41,323,1602.66,2644.93
3,28,336,4237.93,7255.99
4,41,323,1294.39,2302.29
5,41,323,858.02,1642.59
6,29,335,1640.86,3085.97
7,55,309,2358.75,3584.98
8,30,334,2186.27,3921.87
9,90,274,1837.24,3124.71
10,182,182,725.04,1061.93


## 5. Transactions

Join on both `date` and `store_nbr`. Use a single product family to avoid mixing sales units, and calculate store-level correlations.

In [0]:
%sql

WITH daily_sales AS (
    SELECT
        date,
        store_nbr,
        SUM(sales) AS total_sales
    FROM workspace.default.sales_cleaned
    GROUP BY date, store_nbr
)

SELECT
    s.date,
    s.store_nbr,
    ROUND(s.total_sales, 2) AS total_sales,
    t.transactions
FROM daily_sales AS s
INNER JOIN workspace.default.transactions_cleaned AS t
    ON s.date = t.date
    AND s.store_nbr = t.store_nbr
ORDER BY s.date, s.store_nbr
LIMIT 10;


date,store_nbr,total_sales,transactions
2013-01-01,25,2511.62,770
2013-01-02,1,7417.15,2111
2013-01-02,2,10266.72,2358
2013-01-02,3,24060.35,3487
2013-01-02,4,10200.08,1922
2013-01-02,5,10598.62,1903
2013-01-02,6,13520.49,2143
2013-01-02,7,11997.5,1874
2013-01-02,8,14659.33,3250
2013-01-02,9,15867.48,2940


In [0]:
%sql

SELECT
    s.date,
    s.sales AS beverage_sales,
    t.transactions
FROM workspace.default.sales_cleaned AS s
INNER JOIN workspace.default.transactions_cleaned AS t
    ON s.date = t.date
    AND s.store_nbr = t.store_nbr
WHERE s.family = 'BEVERAGES'
    AND s.store_nbr = 1
    AND s.year = 2016
ORDER BY s.date;


date,beverage_sales,transactions
2016-01-04,3005.0,10
2016-01-05,2374.0,1840
2016-01-06,2799.0,1996
2016-01-07,2248.0,1822
2016-01-08,2208.0,1823
2016-01-09,2360.0,1331
2016-01-10,991.0,507
2016-01-11,2001.0,1767
2016-01-12,2339.0,1744
2016-01-13,2141.0,1729


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    COUNT(*) AS matched_days,
    ROUND(
        CORR(t.transactions, s.sales), 3
    ) AS pearson_correlation
FROM workspace.default.sales_cleaned AS s
INNER JOIN workspace.default.transactions_cleaned AS t
    ON s.date = t.date
    AND s.store_nbr = t.store_nbr
WHERE s.family = 'BEVERAGES'
    AND s.store_nbr = 1
    AND s.year = 2016;


matched_days,pearson_correlation
362,0.762


In [0]:
%sql

SELECT
    s.store_nbr,
    COUNT(*) AS matched_days,
    ROUND(CORR(t.transactions, s.sales), 3) AS correlation
FROM workspace.default.sales_cleaned AS s
INNER JOIN workspace.default.transactions_cleaned AS t
    ON s.date = t.date
    AND s.store_nbr = t.store_nbr
WHERE s.family = 'BEVERAGES'
    AND s.year = 2016
GROUP BY s.store_nbr
HAVING COUNT(*) >= 300
ORDER BY correlation DESC;


store_nbr,matched_days,correlation
53,362,0.896
48,362,0.884
28,362,0.847
31,363,0.827
46,362,0.825
50,362,0.807
29,362,0.803
27,362,0.8
19,361,0.782
42,362,0.778


## 6. Holidays

Check holiday types and regional coverage, then build one holiday flag per store and date before joining sales.

In [0]:

df_holidays = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/retail_raw/holidays_events.csv")
)

df_holidays.printSchema()
display(df_holidays.limit(10))


root
 |-- date: date (nullable = true)
 |-- type: string (nullable = true)
 |-- locale: string (nullable = true)
 |-- locale_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- transferred: boolean (nullable = true)



date,type,locale,locale_name,description,transferred
2012-03-02,Holiday,Local,Manta,Fundacion de Manta,false
2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,false
2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,false
2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,false
2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,false
2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,false
2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,false
2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,false
2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,false
2012-06-25,Holiday,Local,Machala,Fundacion de Machala,false


In [0]:
display(
    df_holidays
    .filter("date BETWEEN '2013-01-01' AND '2017-08-15'")
    .groupBy("date")
    .count()
    .filter("count > 1")
    .orderBy("date")
    .limit(10)
)


date,count
2013-05-12,2
2013-06-25,3
2013-07-03,2
2013-12-22,2
2014-06-25,4
2014-07-03,2
2014-12-22,2
2014-12-26,2
2015-06-25,3
2015-07-03,2


In [0]:

display(
    df_holidays
    .filter("date = '2014-06-25'")
    .orderBy("locale", "locale_name")
)


date,type,locale,locale_name,description,transferred
2014-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,false
2014-06-25,Holiday,Local,Machala,Fundacion de Machala,false
2014-06-25,Event,National,Ecuador,Mundial de futbol Brasil: Ecuador-Francia,false
2014-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,false


In [0]:
display(
    df_holidays
    .groupBy("type", "transferred")
    .count()
    .orderBy("type", "transferred")
)


type,transferred,count
Additional,false,51
Bridge,false,5
Event,false,56
Holiday,false,209
Holiday,true,12
Transfer,false,12
Work Day,false,5


In [0]:
display(
    df_holidays
    .filter(
        "(type = 'Holiday' AND transferred = true) OR type = 'Transfer'"
    )
    .orderBy("date", "type")
)


date,type,locale,locale_name,description,transferred
2012-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,true
2012-10-12,Transfer,National,Ecuador,Traslado Independencia de Guayaquil,false
2013-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,true
2013-10-11,Transfer,National,Ecuador,Traslado Independencia de Guayaquil,false
2014-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,true
2014-10-10,Transfer,National,Ecuador,Traslado Independencia de Guayaquil,false
2016-05-24,Holiday,National,Ecuador,Batalla de Pichincha,true
2016-05-27,Transfer,National,Ecuador,Traslado Batalla de Pichincha,false
2016-07-24,Transfer,Local,Guayaquil,Traslado Fundacion de Guayaquil,false
2016-07-25,Holiday,Local,Guayaquil,Fundacion de Guayaquil,true


In [0]:
display(
    df_holidays
    .filter("type IN ('Additional', 'Bridge', 'Work Day')")
    .orderBy("date")
    .limit(20)
)

date,type,locale,locale_name,description,transferred
2012-12-05,Additional,Local,Quito,Fundacion de Quito-1,false
2012-12-21,Additional,National,Ecuador,Navidad-4,false
2012-12-22,Additional,National,Ecuador,Navidad-3,false
2012-12-23,Additional,National,Ecuador,Navidad-2,false
2012-12-24,Bridge,National,Ecuador,Puente Navidad,false
2012-12-24,Additional,National,Ecuador,Navidad-1,false
2012-12-26,Additional,National,Ecuador,Navidad+1,false
2012-12-31,Bridge,National,Ecuador,Puente Primer dia del ano,false
2012-12-31,Additional,National,Ecuador,Primer dia del ano-1,false
2013-01-05,Work Day,National,Ecuador,Recupero puente Navidad,false


In [0]:

df_valid_holidays = df_holidays.filter(
    "(type = 'Holiday' AND transferred = false) "
    "OR type IN ('Additional', 'Bridge', 'Transfer')"
)

display(
    df_valid_holidays
    .groupBy("type")
    .count()
    .orderBy("type")
)


type,count
Additional,51
Bridge,5
Holiday,209
Transfer,12


In [0]:
from pyspark.sql import functions as F

df_store_holiday_matches = (
    df_valid_holidays.alias("h")
    .join(
        spark.table("workspace.default.stores_cleaned").alias("st"),
        F.expr("""
            h.locale = 'National'
            OR (h.locale = 'Regional' AND h.locale_name = st.state)
            OR (h.locale = 'Local' AND h.locale_name = st.city)
        """),
        "inner"
    )
    .select(
        F.col("h.date").alias("date"),
        F.col("st.store_nbr"),
        F.col("st.city"),
        F.col("st.state"),
        F.col("h.type").alias("holiday_type")
    )
)

display(
    df_store_holiday_matches
    .filter("date = '2014-06-25'")
    .orderBy("store_nbr")
)


date,store_nbr,city,state,holiday_type
2014-06-25,12,Latacunga,Cotopaxi,Holiday
2014-06-25,13,Latacunga,Cotopaxi,Holiday
2014-06-25,15,Ibarra,Imbabura,Holiday
2014-06-25,40,Machala,El Oro,Holiday
2014-06-25,41,Machala,El Oro,Holiday


In [0]:

df_holiday_flags = (
    df_store_holiday_matches
    .select("date", "store_nbr")
    .distinct()
    .withColumn("is_holiday", F.lit(True))
)

display(
    df_holiday_flags
    .filter("date = '2014-06-25'")
    .orderBy("store_nbr")
)


date,store_nbr,is_holiday
2014-06-25,12,true
2014-06-25,13,true
2014-06-25,15,true
2014-06-25,40,true
2014-06-25,41,true


In [0]:

df_beverage_holiday = (
    spark.table("workspace.default.sales_cleaned")
    .filter(
        (F.col("family") == "BEVERAGES") &
        (F.col("year").between(2014, 2016))
    )
    .join(
        df_holiday_flags,
        on=["date", "store_nbr"],
        how="left"
    )
    .fillna({"is_holiday": False})
)

display(
    df_beverage_holiday
    .groupBy("is_holiday")
    .agg(
        F.count("*").alias("record_count"),
        F.round(F.avg("sales"), 2).alias("avg_beverage_sales")
    )
    .orderBy("is_holiday")
)


is_holiday,record_count,avg_beverage_sales
false,55901,2537.28
true,3121,3007.91


## 7. Oil prices

Check coverage, align prices to sales dates, and forward-fill missing dates with the latest known oil price.

In [0]:

%sql

SELECT
    MIN(date) AS first_date,
    MAX(date) AS last_date,
    COUNT(*) AS total_records,
    COUNT(dcoilwtico) AS valid_prices,
    COUNT(*) - COUNT(dcoilwtico) AS missing_prices
FROM workspace.default.oil_cleaned;


first_date,last_date,total_records,valid_prices,missing_prices
2013-01-01,2017-08-31,1218,1217,1


In [0]:
%sql

WITH daily_beverages AS (
    SELECT
        date,
        SUM(sales) AS beverage_sales
    FROM workspace.default.sales_cleaned
    WHERE family = 'BEVERAGES'
    GROUP BY date
)

SELECT
    COUNT(*) AS sales_days,
    COUNT(o.date) AS matched_oil_dates,
    COUNT(o.dcoilwtico) AS valid_oil_prices,
    COUNT(*) - COUNT(o.date) AS missing_oil_dates
FROM daily_beverages AS s
LEFT JOIN workspace.default.oil_cleaned AS o
    ON s.date = o.date;


sales_days,matched_oil_dates,valid_oil_prices,missing_oil_dates
1684,1203,1202,481


In [0]:

%sql

WITH daily_beverages AS (
    SELECT
        date,
        SUM(sales) AS beverage_sales
    FROM workspace.default.sales_cleaned
    WHERE family = 'BEVERAGES'
    GROUP BY date
),

daily_aligned AS (
    SELECT
        s.date,
        s.beverage_sales,
        o.dcoilwtico AS original_oil_price,
        LAST_VALUE(o.dcoilwtico, TRUE) OVER (
            ORDER BY s.date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS oil_price_filled
    FROM daily_beverages AS s
    LEFT JOIN workspace.default.oil_cleaned AS o
        ON s.date = o.date
)

SELECT
    COUNT(oil_price_filled) AS paired_days,
    ROUND(
        CORR(oil_price_filled, beverage_sales), 3
    ) AS oil_sales_correlation
FROM daily_aligned;



paired_days,oil_sales_correlation
1683,-0.628


## Key findings

- **Sales patterns:** December had above-average monthly sales indices in all four complete years (2013–2016). Growth and seasonality may overlap.
- **Stores and products:** GROCERY I and BEVERAGES led recorded sales quantities. Quito ranked highest in average daily beverage sales per store.
- **Promotions:** Beverage sales averaged **3,215.50** with promotions versus **1,292.27** without. The difference also appeared within individual stores.
- **Transactions:** Daily transactions and beverage sales were positively correlated in all **51** stores with at least 300 matched days in 2016 (`r` = **0.341–0.896**).
- **Holidays:** Matched holidays had **18.5%** higher average beverage sales than other dates in 2014–2016 (**3,007.91** vs **2,537.28**).
- **Oil prices:** Oil prices and daily beverage sales were negatively correlated (`r` = **-0.628**, 1,683 days).

**Limitations:** These are descriptive associations, not causal effects. Store differences, promotions and long-term trends may influence the results. Different product families can use different sales units.